In [16]:
# Cell 1 — Install dependencies
# fasttext-wheel is intentionally not installed: its native extension can fail
# to compile on newer Kaggle Python versions. Cell 6 implements the FastText
# supervised architecture directly in PyTorch instead.
!pip -q install transformers accelerate

In [17]:
# Cell 2 — Imports and reproducibility
import os, re, random, time, warnings
from collections import Counter
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [18]:
# Cell 3 — Build a small labelled corpus from scratch
# Each language has a word bank and natural-language sentence patterns.
# Random combinations create varied examples without an external dataset.

LANG_DATA = {
    "Hindi": {
        "words": ["मैं", "आप", "हम", "आज", "कल", "घर", "स्कूल", "बाजार", "किताब", "पानी",
                  "खाना", "अच्छा", "सुंदर", "नया", "बड़ा", "छोटा", "सीखना", "पढ़ना", "लिखना",
                  "जाना", "आना", "काम", "भाषा", "मित्र", "परिवार", "समय", "मौसम", "कंप्यूटर"],
        "templates": ["{a} {b} {c} है।", "{a} {b} जाना चाहता है।", "आज {a} {b} {c}।",
                      "मुझे {a} {b} पसंद है।", "{a} और {b} बहुत {c} हैं।",
                      "हम {a} में {b} करते हैं।", "क्या आप {a} {b} सकते हैं?"]
    },
    "Gujarati": {
        "words": ["હું", "તમે", "અમે", "આજે", "કાલે", "ઘર", "શાળા", "બજાર", "પુસ્તક", "પાણી",
                  "ખોરાક", "સારું", "સુંદર", "નવું", "મોટું", "નાનું", "શીખવું", "વાંચવું", "લખવું",
                  "જવું", "આવવું", "કામ", "ભાષા", "મિત્ર", "પરિવાર", "સમય", "હવામાન", "કોમ્પ્યુટર"],
        "templates": ["{a} {b} {c} છે.", "{a} {b} જવા માંગે છે.", "આજે {a} {b} {c}.",
                      "મને {a} {b} ગમે છે.", "{a} અને {b} ખૂબ {c} છે.",
                      "અમે {a} માં {b} કરીએ છીએ.", "શું તમે {a} {b} શકો છો?"]
    },
    "Bengali": {
        "words": ["আমি", "আপনি", "আমরা", "আজ", "কাল", "বাড়ি", "স্কুল", "বাজার", "বই", "জল",
                  "খাবার", "ভালো", "সুন্দর", "নতুন", "বড়", "ছোট", "শেখা", "পড়া", "লেখা",
                  "যাওয়া", "আসা", "কাজ", "ভাষা", "বন্ধু", "পরিবার", "সময়", "আবহাওয়া", "কম্পিউটার"],
        "templates": ["{a} {b} {c}।", "{a} {b} যেতে চায়।", "আজ {a} {b} {c}।",
                      "আমার {a} {b} ভালো লাগে।", "{a} এবং {b} খুব {c}।",
                      "আমরা {a} তে {b} করি।", "আপনি কি {a} {b} পারেন?"]
    },
    "Marathi": {
        "words": ["मी", "तुम्ही", "आम्ही", "आज", "उद्या", "घर", "शाळा", "बाजार", "पुस्तक", "पाणी",
                  "जेवण", "चांगले", "सुंदर", "नवीन", "मोठे", "लहान", "शिकणे", "वाचणे", "लिहिणे",
                  "जाणे", "येणे", "काम", "भाषा", "मित्र", "कुटुंब", "वेळ", "हवामान", "संगणक"],
        "templates": ["{a} {b} {c} आहे.", "{a} {b} जायचे आहे.", "आज {a} {b} {c}.",
                      "मला {a} {b} आवडते.", "{a} आणि {b} खूप {c} आहेत.",
                      "आम्ही {a} मध्ये {b} करतो.", "तुम्ही {a} {b} शकता का?"]
    },
    "Tamil": {
        "words": ["நான்", "நீங்கள்", "நாங்கள்", "இன்று", "நாளை", "வீடு", "பள்ளி", "சந்தை", "புத்தகம்", "தண்ணீர்",
                  "உணவு", "நல்ல", "அழகான", "புதிய", "பெரிய", "சிறிய", "கற்றல்", "படித்தல்", "எழுதுதல்",
                  "செல்ல", "வர", "வேலை", "மொழி", "நண்பர்", "குடும்பம்", "நேரம்", "வானிலை", "கணினி"],
        "templates": ["{a} {b} {c} ஆகும்.", "{a} {b} செல்ல விரும்புகிறேன்.", "இன்று {a} {b} {c}.",
                      "எனக்கு {a} {b} பிடிக்கும்.", "{a} மற்றும் {b} மிகவும் {c}.",
                      "நாங்கள் {a} இல் {b} செய்கிறோம்.", "நீங்கள் {a} {b} முடியுமா?"]
    },
    "Telugu": {
        "words": ["నేను", "మీరు", "మేము", "ఈరోజు", "రేపు", "ఇల్లు", "పాఠశాల", "బజారు", "పుస్తకం", "నీరు",
                  "ఆహారం", "మంచి", "అందమైన", "కొత్త", "పెద్ద", "చిన్న", "నేర్చుకోవడం", "చదవడం", "రాయడం",
                  "వెళ్ళడం", "రావడం", "పని", "భాష", "స్నేహితుడు", "కుటుంబం", "సమయం", "వాతావరణం", "కంప్యూటర్"],
        "templates": ["{a} {b} {c} ఉంది.", "{a} {b} వెళ్లాలనుకుంటున్నారు.", "ఈరోజు {a} {b} {c}.",
                      "నాకు {a} {b} ఇష్టం.", "{a} మరియు {b} చాలా {c}.",
                      "మేము {a} లో {b} చేస్తాము.", "మీరు {a} {b} చేయగలరా?"]
    }
}

def make_examples(lang, n=1200):
    cfg = LANG_DATA[lang]
    out = []
    for _ in range(n):
        template = random.choice(cfg["templates"])
        a, b, c = random.sample(cfg["words"], 3)
        out.append(template.format(a=a, b=b, c=c))
    return out

rows = []
for language in LANG_DATA:
    for sentence in make_examples(language, n=1200):
        rows.append({"text": sentence, "label": language})

df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Dataset shape:", df.shape)
print(df.groupby("label").size())
display(df.sample(10, random_state=SEED))

# Save a copy so the generated dataset is available as a Kaggle output file.
os.makedirs("/kaggle/working", exist_ok=True)
df.to_csv("/kaggle/working/indian_language_identification_dataset.csv", index=False, encoding="utf-8")


Dataset shape: (7200, 2)
label
Bengali     1200
Gujarati    1200
Hindi       1200
Marathi     1200
Tamil       1200
Telugu      1200
dtype: int64


,text,label
3098,नया बड़ा भाषा है।,Hindi
2531,हम जाना में आज करते हैं।,Hindi
4071,શું તમે હું વાંચવું શકો છો?,Gujarati
1287,આજે નવું આવવું ખોરાક.,Gujarati
2540,ভালো এবং আজ খুব আপনি।,Bengali
263,आज घर काम मित्र.,Marathi
6123,আমার জল ভাষা ভালো লাগে।,Bengali
2439,खाना और कंप्यूटर बहुत आज हैं।,Hindi
3749,कल और खाना बहुत कंप्यूटर हैं।,Hindi
1199,రాయడం సమయం వెళ్లాలనుకుంటున్నారు.,Telugu


In [19]:
# Cell 4 — Train/test split and label encoding
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
class_names = list(label_encoder.classes_)
num_classes = len(class_names)

X_train, X_test, y_train, y_test = train_test_split(
    df["text"].tolist(),
    df["label_id"].to_numpy(),
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_id"]
)
print("Classes:", class_names)
print("Train:", len(X_train), "Test:", len(X_test))

def evaluate_predictions(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    print(f"\\n{model_name} — accuracy: {acc:.4f}, macro-F1: {macro:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    return {"Model": model_name, "Accuracy": acc, "Macro F1": macro}

results = []


Classes: ['Bengali', 'Gujarati', 'Hindi', 'Marathi', 'Tamil', 'Telugu']
Train: 5760 Test: 1440


In [20]:
# Cell 5 — Logistic Regression and MLP using character TF-IDF
# Character n-grams are useful for language identification because scripts and spelling patterns differ.

tfidf = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), min_df=2, max_features=50000)
Xtr_vec = tfidf.fit_transform(X_train)
Xte_vec = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000, C=5)
lr.fit(Xtr_vec, y_train)
pred_lr = lr.predict(Xte_vec)
results.append(evaluate_predictions("Logistic Regression (char TF-IDF)", y_test, pred_lr))

mlp = MLPClassifier(hidden_layer_sizes=(256, 128), activation="relu",
                    max_iter=25, early_stopping=True, random_state=SEED)
mlp.fit(Xtr_vec, y_train)
pred_mlp = mlp.predict(Xte_vec)
results.append(evaluate_predictions("MLP (char TF-IDF)", y_test, pred_mlp))


\nLogistic Regression (char TF-IDF) — accuracy: 1.0000, macro-F1: 1.0000
              precision    recall  f1-score   support

     Bengali       1.00      1.00      1.00       240
    Gujarati       1.00      1.00      1.00       240
       Hindi       1.00      1.00      1.00       240
     Marathi       1.00      1.00      1.00       240
       Tamil       1.00      1.00      1.00       240
      Telugu       1.00      1.00      1.00       240

    accuracy                           1.00      1440
   macro avg       1.00      1.00      1.00      1440
weighted avg       1.00      1.00      1.00      1440

\nMLP (char TF-IDF) — accuracy: 1.0000, macro-F1: 1.0000
              precision    recall  f1-score   support

     Bengali       1.00      1.00      1.00       240
    Gujarati       1.00      1.00      1.00       240
       Hindi       1.00      1.00      1.00       240
     Marathi       1.00      1.00      1.00       240
       Tamil       1.00      1.00      1.00       240
  

In [21]:
# Cell 6 — FastText-style supervised classifier (pure PyTorch; no fasttext package)
# FastText represents a document by averaging word and character-subword n-gram
# embeddings, then feeds that representation to a linear classification layer.
# This avoids the fasttext-wheel build/NumPy compatibility errors on Kaggle.

from collections import Counter
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

def tokenize_words(text):
    return re.findall(r"\w+", str(text).lower(), flags=re.UNICODE)

def make_subword_features(text, min_n=3, max_n=5):
    # Word tokens plus character n-grams with boundary markers (FastText idea).
    features = []
    for word in tokenize_words(text):
        features.append("W:" + word)
        marked = "<" + word + ">"
        for n in range(min_n, max_n + 1):
            for i in range(max(0, len(marked) - n + 1)):
                features.append("C:" + marked[i:i+n])
    return features or ["W:<empty>"]

# Build feature vocabulary from training data only.
feature_counts = Counter()
for text in X_train:
    feature_counts.update(make_subword_features(text))

MIN_FEATURE_FREQ = 2
feature_to_id = {
    feature: i + 1
    for i, (feature, count) in enumerate(
        (item for item in feature_counts.items() if item[1] >= MIN_FEATURE_FREQ)
    )
}
FASTTEXT_UNK = 0
FASTTEXT_MAX_FEATURES = 500

def encode_fasttext(text):
    ids = [feature_to_id.get(f, FASTTEXT_UNK) for f in make_subword_features(text)]
    ids = [i for i in ids if i != FASTTEXT_UNK]
    if not ids:
        ids = [FASTTEXT_UNK]
    return torch.tensor(ids[:FASTTEXT_MAX_FEATURES], dtype=torch.long)

class FastTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.x = [encode_fasttext(t) for t in texts]
        self.y = [int(y) for y in labels]
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

def collate_fasttext(batch):
    xs, ys = zip(*batch)
    flat = torch.cat(xs)
    offsets = [0]
    for x in xs:
        offsets.append(offsets[-1] + len(x))
    return flat, torch.tensor(offsets[:-1], dtype=torch.long), torch.tensor(ys, dtype=torch.long)

ft_train_dl = DataLoader(
    FastTextDataset(X_train, y_train), batch_size=128, shuffle=True,
    collate_fn=collate_fasttext
)
ft_test_dl = DataLoader(
    FastTextDataset(X_test, y_test), batch_size=256, shuffle=False,
    collate_fn=collate_fasttext
)

class FastTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_classes):
        super().__init__()
        self.embedding = nn.EmbeddingBag(
            vocab_size + 1, embedding_dim, mode="mean", padding_idx=FASTTEXT_UNK
        )
        self.dropout = nn.Dropout(0.2)
        self.linear = nn.Linear(embedding_dim, n_classes)
    def forward(self, token_ids, offsets):
        pooled = self.embedding(token_ids, offsets)
        return self.linear(self.dropout(pooled))

ft_model = FastTextClassifier(len(feature_to_id), 100, num_classes).to(DEVICE)
ft_optimizer = torch.optim.AdamW(ft_model.parameters(), lr=1e-3)
ft_loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    ft_model.train()
    running_loss = 0.0
    for token_ids, offsets, labels in ft_train_dl:
        token_ids, offsets, labels = token_ids.to(DEVICE), offsets.to(DEVICE), labels.to(DEVICE)
        ft_optimizer.zero_grad()
        logits = ft_model(token_ids, offsets)
        loss = ft_loss_fn(logits, labels)
        loss.backward()
        ft_optimizer.step()
        running_loss += loss.item()
    print(f"FastText-style epoch {epoch+1}/8 — loss {running_loss/len(ft_train_dl):.4f}")

ft_model.eval()
pred_ft = []
with torch.no_grad():
    for token_ids, offsets, labels in ft_test_dl:
        logits = ft_model(token_ids.to(DEVICE), offsets.to(DEVICE))
        pred_ft.extend(logits.argmax(1).cpu().numpy())

pred_ft = np.array(pred_ft)
results.append(evaluate_predictions("FastText-style (word + subword n-grams)", y_test, pred_ft))


FastText-style epoch 1/8 — loss 1.5644
FastText-style epoch 2/8 — loss 1.0747
FastText-style epoch 3/8 — loss 0.6026
FastText-style epoch 4/8 — loss 0.3028
FastText-style epoch 5/8 — loss 0.1629
FastText-style epoch 6/8 — loss 0.0977
FastText-style epoch 7/8 — loss 0.0658
FastText-style epoch 8/8 — loss 0.0477
\nFastText-style (word + subword n-grams) — accuracy: 0.9986, macro-F1: 0.9986
              precision    recall  f1-score   support

     Bengali       1.00      1.00      1.00       240
    Gujarati       1.00      1.00      1.00       240
       Hindi       0.99      1.00      1.00       240
     Marathi       1.00      0.99      1.00       240
       Tamil       1.00      1.00      1.00       240
      Telugu       1.00      1.00      1.00       240

    accuracy                           1.00      1440
   macro avg       1.00      1.00      1.00      1440
weighted avg       1.00      1.00      1.00      1440



In [22]:
# Cell 7 — Character-level vocabulary for LSTM and GRU
# Character-level models avoid needing a separate tokenizer for each Indian script.
from torch.nn.utils.rnn import pad_sequence

PAD, UNK = 0, 1
all_chars = sorted(set("".join(X_train)))
char_to_id = {ch: i + 2 for i, ch in enumerate(all_chars)}
vocab_size = len(char_to_id) + 2
MAX_CHARS = 160

def encode_chars(text):
    ids = [char_to_id.get(ch, UNK) for ch in text[:MAX_CHARS]]
    return torch.tensor(ids if ids else [UNK], dtype=torch.long)

class CharDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return encode_chars(self.texts[idx]), int(self.labels[idx])

def collate_char(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    xpad = pad_sequence(xs, batch_first=True, padding_value=PAD)
    return xpad, lengths, torch.tensor(ys, dtype=torch.long)

train_char_ds = CharDataset(X_train, y_train)
test_char_ds = CharDataset(X_test, y_test)
train_char_dl = DataLoader(train_char_ds, batch_size=128, shuffle=True, collate_fn=collate_char)
test_char_dl = DataLoader(test_char_ds, batch_size=256, shuffle=False, collate_fn=collate_char)

class CharRNNClassifier(nn.Module):
    def __init__(self, rnn_type="LSTM", emb_dim=64, hidden=128, n_classes=num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        rnn_cls = nn.LSTM if rnn_type == "LSTM" else nn.GRU
        self.rnn = rnn_cls(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden * 2, n_classes)
    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.rnn(packed)
        if isinstance(hidden, tuple):
            hidden = hidden[0]
        # Last layer's forward and backward hidden states
        features = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(features))

def train_rnn(model, epochs=8):
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, lengths, yb in train_char_dl:
            xb, lengths, yb = xb.to(DEVICE), lengths.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb, lengths)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} — loss {total_loss/len(train_char_dl):.4f}")
    return model

def predict_rnn(model):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, lengths, _ in test_char_dl:
            logits = model(xb.to(DEVICE), lengths.to(DEVICE))
            preds.extend(logits.argmax(1).cpu().numpy())
    return np.array(preds)

lstm_model = train_rnn(CharRNNClassifier("LSTM"), epochs=8)
pred_lstm = predict_rnn(lstm_model)
results.append(evaluate_predictions("BiLSTM (character-level)", y_test, pred_lstm))

gru_model = train_rnn(CharRNNClassifier("GRU"), epochs=8)
pred_gru = predict_rnn(gru_model)
results.append(evaluate_predictions("BiGRU (character-level)", y_test, pred_gru))


Epoch 1/8 — loss 0.6551
Epoch 2/8 — loss 0.0046
Epoch 3/8 — loss 0.0017
Epoch 4/8 — loss 0.0009
Epoch 5/8 — loss 0.0006
Epoch 6/8 — loss 0.0004
Epoch 7/8 — loss 0.0003
Epoch 8/8 — loss 0.0003
\nBiLSTM (character-level) — accuracy: 1.0000, macro-F1: 1.0000
              precision    recall  f1-score   support

     Bengali       1.00      1.00      1.00       240
    Gujarati       1.00      1.00      1.00       240
       Hindi       1.00      1.00      1.00       240
     Marathi       1.00      1.00      1.00       240
       Tamil       1.00      1.00      1.00       240
      Telugu       1.00      1.00      1.00       240

    accuracy                           1.00      1440
   macro avg       1.00      1.00      1.00      1440
weighted avg       1.00      1.00      1.00      1440

Epoch 1/8 — loss 0.5191
Epoch 2/8 — loss 0.0049
Epoch 3/8 — loss 0.0018
Epoch 4/8 — loss 0.0011
Epoch 5/8 — loss 0.0008
Epoch 6/8 — loss 0.0006
Epoch 7/8 — loss 0.0004
Epoch 8/8 — loss 0.0004
\nBiGRU (

In [23]:
# Cell 8 — Fine-tune MuRIL (BERT-family model for Indian languages)
# Internet must be enabled in Kaggle to download the pretrained checkpoint.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

BERT_MODEL = "google/muril-base-cased"
MAX_LENGTH = 128
BERT_BATCH_SIZE = 16
BERT_EPOCHS = 2

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=num_classes
).to(DEVICE)

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]), truncation=True, padding="max_length",
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

bert_train_dl = DataLoader(TextDataset(X_train, y_train), batch_size=BERT_BATCH_SIZE, shuffle=True)
bert_test_dl = DataLoader(TextDataset(X_test, y_test), batch_size=BERT_BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)
num_training_steps = len(bert_train_dl) * BERT_EPOCHS
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1, total_iters=max(1, num_training_steps)
)

for epoch in range(BERT_EPOCHS):
    bert_model.train()
    total_loss = 0
    for batch in bert_train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        output = bert_model(**batch)
        loss = output.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"MuRIL epoch {epoch+1}/{BERT_EPOCHS} — loss {total_loss/len(bert_train_dl):.4f}")

bert_model.eval()
pred_bert = []
with torch.no_grad():
    for batch in bert_test_dl:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
        pred_bert.extend(logits.argmax(1).cpu().numpy())

pred_bert = np.array(pred_bert)
results.append(evaluate_predictions("MuRIL fine-tuned", y_test, pred_bert))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

MuRIL epoch 1/2 — loss 1.4602
MuRIL epoch 2/2 — loss 0.9291
\nMuRIL fine-tuned — accuracy: 1.0000, macro-F1: 1.0000
              precision    recall  f1-score   support

     Bengali       1.00      1.00      1.00       240
    Gujarati       1.00      1.00      1.00       240
       Hindi       1.00      1.00      1.00       240
     Marathi       1.00      1.00      1.00       240
       Tamil       1.00      1.00      1.00       240
      Telugu       1.00      1.00      1.00       240

    accuracy                           1.00      1440
   macro avg       1.00      1.00      1.00      1440
weighted avg       1.00      1.00      1.00      1440



In [24]:
# Cell 9 — Compare all models and save results
results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False).reset_index(drop=True)
display(results_df.style.format({"Accuracy": "{:.4f}", "Macro F1": "{:.4f}"}))
results_df.to_csv("/kaggle/working/model_comparison_results.csv", index=False)
print("Saved dataset and comparison table in /kaggle/working/")


,Model,Accuracy,Macro F1
0,Logistic Regression (char TF-IDF),1.0000,1.0000
1,MLP (char TF-IDF),1.0000,1.0000
2,BiLSTM (character-level),1.0000,1.0000
3,BiGRU (character-level),1.0000,1.0000
4,MuRIL fine-tuned,1.0000,1.0000
5,FastText-style (word + subword n-grams),0.9986,0.9986


Saved dataset and comparison table in /kaggle/working/


In [25]:
# Cell 10 — Try the fine-tuned MuRIL model on your own text
def identify_language(text):
    bert_model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(probs))
    return class_names[idx], float(probs[idx])

examples = [
    "આજે હું શાળાએ જાઉં છું.",
    "आज मौसम अच्छा है।",
    "আমি বাংলা ভাষা শিখছি।"
]
for sentence in examples:
    print(sentence, "->", identify_language(sentence))


આજે હું શાળાએ જાઉં છું. -> ('Gujarati', 0.4625406563282013)
आज मौसम अच्छा है। -> ('Hindi', 0.4629802703857422)
আমি বাংলা ভাষা শিখছি। -> ('Bengali', 0.44551366567611694)
